[Reference](https://levelup.gitconnected.com/your-production-ai-app-needs-an-llm-router-not-one-favorite-model-5c0fac909fa1$0)

# 1. Define the core types

In [1]:
from __future__ import annotations

import asyncio
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Protocol

class TaskType(str, Enum):
    CHAT = "chat"
    REWRITE = "rewrite"
    EXTRACTION = "extraction"
    RAG = "rag"
    REASONING = "reasoning"
    CODE = "code"
    VISION = "vision"

class RiskLevel(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

@dataclass(frozen=True)
class LLMRequest:
    user_id: str
    input_text: str
    product_surface: str
    required_json: bool = False
    has_image: bool = False
    max_latency_ms: int = 4000
    max_cost_cents: float | None = None
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class RequestProfile:
    task_type: TaskType
    risk_level: RiskLevel
    input_tokens_estimate: int
    needs_long_context: bool
    needs_tools: bool
    required_json: bool
    has_image: bool

@dataclass(frozen=True)
class ModelRoute:
    name: str
    provider: str
    model: str
    max_input_tokens: int
    expected_latency_ms: int
    input_cost_per_1m: float
    output_cost_per_1m: float
    supports_json: bool = True
    supports_vision: bool = False
    supports_tools: bool = True
    fallback_routes: tuple[str, ...] = ()

@dataclass
class LLMResponse:
    text: str
    route_name: str
    model: str
    latency_ms: int
    input_tokens_estimate: int
    output_tokens_estimate: int
    attempts: int
    metadata: dict[str, Any] = field(default_factory=dict)

# 2. Create provider adapters


In [2]:
class ModelClient(Protocol):
    async def generate(
        self,
        route: ModelRoute,
        request: LLMRequest,
        profile: RequestProfile,
    ) -> str:
        ...

class ExampleProviderClient:
    async def generate(
        self,
        route: ModelRoute,
        request: LLMRequest,
        profile: RequestProfile,
    ) -> str:
        # Replace this method with your provider SDK call.
        # Keep prompts and provider details inside this adapter.
        await asyncio.sleep(route.expected_latency_ms / 1000)
        if profile.required_json:
            return '{"answer": "example", "route": "%s"}' % route.name
        return f"Example answer from {route.model}: {request.input_text[:120]}"

In [4]:
clients = {
    "fast_provider": FastProviderClient(api_key=...),
    "frontier_provider": FrontierProviderClient(api_key=...),
    "local": LocalVLLMClient(base_url="http://llm-inference:8000"),
}

# 3. Analyze the request


In [5]:
class RequestAnalyzer:
    def profile(self, request: LLMRequest) -> RequestProfile:
        text = request.input_text.lower()
        token_estimate = max(1, len(request.input_text) // 4)
        if request.has_image:
            task_type = TaskType.VISION
        elif request.required_json or "extract" in text or "json" in text:
            task_type = TaskType.EXTRACTION
        elif "rewrite" in text or "make this sound" in text:
            task_type = TaskType.REWRITE
        elif "code" in text or "debug" in text or "stack trace" in text:
            task_type = TaskType.CODE
        elif "compare" in text or "reason" in text or "step by step" in text:
            task_type = TaskType.REASONING
        elif request.metadata.get("use_rag") is True:
            task_type = TaskType.RAG
        else:
            task_type = TaskType.CHAT
        high_risk_terms = {
            "legal",
            "medical",
            "diagnosis",
            "investment",
            "refund",
            "contract",
            "security incident",
            "production outage",
        }
        risk = RiskLevel.HIGH if any(term in text for term in high_risk_terms) else RiskLevel.LOW
        if task_type in {TaskType.REASONING, TaskType.CODE, TaskType.RAG}:
            risk = RiskLevel.MEDIUM if risk == RiskLevel.LOW else risk
        return RequestProfile(
            task_type=task_type,
            risk_level=risk,
            input_tokens_estimate=token_estimate,
            needs_long_context=token_estimate > 24000,
            needs_tools=task_type in {TaskType.RAG, TaskType.CODE},
            required_json=request.required_json,
            has_image=request.has_image,
        )

# 4. Build the routing policy


In [6]:
ROUTES: dict[str, ModelRoute] = {
    "cheap_fast": ModelRoute(
        name="cheap_fast",
        provider="fast_provider",
        model="small-fast-model",
        max_input_tokens=32000,
        expected_latency_ms=700,
        input_cost_per_1m=0.10,
        output_cost_per_1m=0.40,
        supports_json=True,
        supports_tools=False,
        fallback_routes=("balanced",),
    ),
    "balanced": ModelRoute(
        name="balanced",
        provider="frontier_provider",
        model="balanced-general-model",
        max_input_tokens=128000,
        expected_latency_ms=1800,
        input_cost_per_1m=1.00,
        output_cost_per_1m=4.00,
        supports_json=True,
        supports_tools=True,
        fallback_routes=("strong_reasoning",),
    ),
    "strong_reasoning": ModelRoute(
        name="strong_reasoning",
        provider="frontier_provider",
        model="large-reasoning-model",
        max_input_tokens=200000,
        expected_latency_ms=5500,
        input_cost_per_1m=5.00,
        output_cost_per_1m=20.00,
        supports_json=True,
        supports_tools=True,
        fallback_routes=("balanced",),
    ),
    "vision": ModelRoute(
        name="vision",
        provider="frontier_provider",
        model="vision-capable-model",
        max_input_tokens=128000,
        expected_latency_ms=2500,
        input_cost_per_1m=2.00,
        output_cost_per_1m=8.00,
        supports_json=True,
        supports_vision=True,
        supports_tools=True,
        fallback_routes=("strong_reasoning",),
    ),
    "local_private": ModelRoute(
        name="local_private",
        provider="local",
        model="self-hosted-private-model",
        max_input_tokens=16000,
        expected_latency_ms=1200,
        input_cost_per_1m=0.03,
        output_cost_per_1m=0.09,
        supports_json=False,
        supports_tools=False,
        fallback_routes=("balanced",),
    ),
}

In [7]:
class RoutingPolicy:
    def __init__(self, routes: dict[str, ModelRoute]) -> None:
        self.routes = routes

    def choose(self, request: LLMRequest, profile: RequestProfile) -> ModelRoute:
        if profile.has_image:
            return self.routes["vision"]
        if profile.needs_long_context:
            return self.routes["strong_reasoning"]
        if request.metadata.get("data_residency") == "private":
            route = self.routes["local_private"]
            if self._route_can_handle(route, request, profile):
                return route
        if profile.risk_level == RiskLevel.HIGH:
            return self.routes["strong_reasoning"]
        if profile.task_type in {TaskType.REASONING, TaskType.CODE, TaskType.RAG}:
            return self.routes["balanced"]
        if profile.task_type in {TaskType.REWRITE, TaskType.EXTRACTION, TaskType.CHAT}:
            cheap = self.routes["cheap_fast"]
            if self._route_can_handle(cheap, request, profile):
                return cheap
        return self.routes["balanced"]

    def _route_can_handle(
        self,
        route: ModelRoute,
        request: LLMRequest,
        profile: RequestProfile,
    ) -> bool:
        if profile.input_tokens_estimate > route.max_input_tokens:
            return False
        if profile.required_json and not route.supports_json:
            return False
        if profile.has_image and not route.supports_vision:
            return False
        if profile.needs_tools and not route.supports_tools:
            return False
        if route.expected_latency_ms > request.max_latency_ms:
            return False
        if request.max_cost_cents is not None:
            estimated_cost = estimate_cost_cents(route, profile.input_tokens_estimate, 800)
            if estimated_cost > request.max_cost_cents:
                return False
        return True

def estimate_cost_cents(
    route: ModelRoute,
    input_tokens: int,
    output_tokens: int,
) -> float:
    input_dollars = (input_tokens / 1_000_000) * route.input_cost_per_1m
    output_dollars = (output_tokens / 1_000_000) * route.output_cost_per_1m
    return (input_dollars + output_dollars) * 100

# 5. Validate outputs before returning them


In [8]:
import json

class ResponseValidator:
    def validate(
        self,
        text: str,
        request: LLMRequest,
        profile: RequestProfile,
    ) -> tuple[bool, str | None]:
        if not text.strip():
            return False, "empty_response"
        if profile.required_json:
            try:
                json.loads(text)
            except json.JSONDecodeError:
                return False, "invalid_json"
        if profile.risk_level == RiskLevel.HIGH and "I am not sure" in text:
            return False, "low_confidence_high_risk"
        if len(text) > 12000 and request.product_surface == "chat_widget":
            return False, "too_long_for_surface"
        return True, None

# 6. Put the router together


In [9]:
class TelemetrySink:
    async def record(self, event: dict[str, Any]) -> None:
        # Replace with OpenTelemetry, Datadog, Honeycomb, Langfuse,
        # Arize Phoenix, or your own trace store.
        print(event)

class LLMRouter:
    def __init__(
        self,
        analyzer: RequestAnalyzer,
        policy: RoutingPolicy,
        validator: ResponseValidator,
        clients: dict[str, ModelClient],
        telemetry: TelemetrySink,
    ) -> None:
        self.analyzer = analyzer
        self.policy = policy
        self.validator = validator
        self.clients = clients
        self.telemetry = telemetry
    async def generate(self, request: LLMRequest) -> LLMResponse:
        profile = self.analyzer.profile(request)
        first_route = self.policy.choose(request, profile)
        route_names = [first_route.name, *first_route.fallback_routes]
        last_error: str | None = None
        started_total = time.perf_counter()
        for attempt, route_name in enumerate(route_names, start=1):
            route = self.policy.routes[route_name]
            if not self.policy._route_can_handle(route, request, profile):
                last_error = "route_cannot_handle_request"
                continue
            started = time.perf_counter()
            try:
                client = self.clients[route.provider]
                text = await client.generate(route, request, profile)
                latency_ms = int((time.perf_counter() - started) * 1000)
                ok, reason = self.validator.validate(text, request, profile)
                await self.telemetry.record(
                    {
                        "event": "llm_route_attempt",
                        "user_id": request.user_id,
                        "task_type": profile.task_type.value,
                        "risk_level": profile.risk_level.value,
                        "route": route.name,
                        "model": route.model,
                        "attempt": attempt,
                        "latency_ms": latency_ms,
                        "valid": ok,
                        "failure_reason": reason,
                    }
                )
                if ok:
                    return LLMResponse(
                        text=text,
                        route_name=route.name,
                        model=route.model,
                        latency_ms=int((time.perf_counter() - started_total) * 1000),
                        input_tokens_estimate=profile.input_tokens_estimate,
                        output_tokens_estimate=max(1, len(text) // 4),
                        attempts=attempt,
                    )
                last_error = reason
            except Exception as exc:
                last_error = exc.__class__.__name__
                await self.telemetry.record(
                    {
                        "event": "llm_route_exception",
                        "user_id": request.user_id,
                        "route": route.name,
                        "model": route.model,
                        "attempt": attempt,
                        "error": last_error,
                    }
                )
        raise RuntimeError(f"All LLM routes failed. Last error: {last_error}")

# 7. Use it from an application


In [10]:
async def main() -> None:
    clients: dict[str, ModelClient] = {
        "fast_provider": ExampleProviderClient(),
        "frontier_provider": ExampleProviderClient(),
        "local": ExampleProviderClient(),
    }

    router = LLMRouter(
            analyzer=RequestAnalyzer(),
            policy=RoutingPolicy(ROUTES),
            validator=ResponseValidator(),
            clients=clients,
            telemetry=TelemetrySink(),
        )
    request = LLMRequest(
        user_id="user_123",
        input_text="Extract the invoice number and total as JSON from this text: INV-8821, total $491.20",
        product_surface="document_ai",
        required_json=True,
        max_latency_ms=3000,
        max_cost_cents=0.2,
    )
    response = await router.generate(request)
    print(response)

if __name__ == "__main__":
    asyncio.run(main())

# Adding evals to routing


In [11]:
async def evaluate_routes(
    router: LLMRouter,
    eval_rows: list[dict[str, Any]],
    ) -> list[dict[str, Any]]:
    results = []

    for row in eval_rows:
        request = LLMRequest(
            user_id="eval",
            input_text=row["input"],
            product_surface="eval",
            required_json=row.get("required_json", False),
            metadata={"expected_route": row.get("expected_route")},
        )
        started = time.perf_counter()
        try:
            response = await router.generate(request)
            passed = response.route_name == row.get("expected_route", response.route_name)
            error = None
        except Exception as exc:
            response = None
            passed = False
            error = exc.__class__.__name__
        results.append(
            {
                "id": row["id"],
                "passed": passed,
                "route": response.route_name if response else None,
                "latency_ms": int((time.perf_counter() - started) * 1000),
                "error": error,
            }
        )
    return results